In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

In [24]:
dataset = pd.read_csv('ai_jobs_salaries_clean.csv')
dataset.head(5)

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size,experience_level_label,employment_type_label,work_mode,salary_outlier_flag,role_family,isco_group_hint
0,2025,EX,FT,Head of Data,348516,USD,348516,US,0,US,M,Executive-level,Full-time,On-site,True,Analytics Manager,Information and communications technology serv...
1,2025,EX,FT,Head of Data,232344,USD,232344,US,0,US,M,Executive-level,Full-time,On-site,False,Analytics Manager,Information and communications technology serv...
2,2025,SE,FT,Data Scientist,145400,USD,145400,US,0,US,M,Senior-level,Full-time,On-site,False,Data Scientist,"Mathematicians, actuaries and statisticians"
3,2025,SE,FT,Data Scientist,81600,USD,81600,US,0,US,M,Senior-level,Full-time,On-site,False,Data Scientist,"Mathematicians, actuaries and statisticians"
4,2025,MI,FT,Engineer,160000,USD,160000,US,100,US,M,Mid-level,Full-time,Remote,False,Other / Unclassified,NaN


In [25]:
dataset['High_Salary'] = (dataset['salary_in_usd'] >= 150000).astype(int)

leak_cols = [
    'High_Salary',
    'salary',
    'salary_currency',
    'salary_in_usd',
    'salary_outlier_flag',
]

In [26]:
print(dataset.isnull().sum())

work_year                     0
experience_level              0
employment_type               0
job_title                     0
salary                        0
salary_currency               0
salary_in_usd                 0
employee_residence            0
remote_ratio                  0
company_location              0
company_size                  0
experience_level_label        0
employment_type_label         0
work_mode                     0
salary_outlier_flag           0
role_family                   0
isco_group_hint           40540
High_Salary                   0
dtype: int64


In [27]:
dataset = dataset.dropna()

In [28]:
datatrain, datatest = train_test_split(dataset, test_size=0.2, shuffle=True)

In [29]:
X_train = datatrain.drop(columns=leak_cols)
y_train = datatrain['High_Salary']
X_test = datatest.drop(columns=leak_cols)
y_test = datatest['High_Salary']

In [30]:
X_train.head(5)

,work_year,experience_level,employment_type,job_title,employee_residence,remote_ratio,company_location,company_size,experience_level_label,employment_type_label,work_mode,role_family,isco_group_hint
23245,2025,SE,FT,Machine Learning Engineer,US,0,US,M,Senior-level,Full-time,On-site,ML Engineer,Software and applications developers
54769,2024,SE,FT,Data Engineer,US,0,US,M,Senior-level,Full-time,On-site,Data Engineer,Software and applications developers
29564,2025,SE,FT,Machine Learning Engineer,US,0,US,M,Senior-level,Full-time,On-site,ML Engineer,Software and applications developers
70903,2022,SE,FT,Data Analyst,US,0,US,M,Senior-level,Full-time,On-site,Data Analyst,Business services and administration managers
52638,2024,SE,FT,Data Scientist,CA,0,CA,M,Senior-level,Full-time,On-site,Data Scientist,"Mathematicians, actuaries and statisticians"


In [31]:
cols = ['experience_level', 'employment_type', 'job_title', 'employee_residence', 'company_location', 
        'company_size', 'experience_level_label', 'employment_type_label', 'work_mode', 'role_family', 'isco_group_hint']

In [32]:
from catboost import CatBoostClassifier
model = CatBoostClassifier(iterations=1000, cat_features=cols, task_type='GPU')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

Learning rate set to 0.028235
0:	learn: 0.6844244	total: 44.3ms	remaining: 44.3s
1:	learn: 0.6762385	total: 90.1ms	remaining: 45s
2:	learn: 0.6684313	total: 148ms	remaining: 49.2s
3:	learn: 0.6616562	total: 208ms	remaining: 51.7s
4:	learn: 0.6546229	total: 260ms	remaining: 51.8s
5:	learn: 0.6479076	total: 307ms	remaining: 50.9s
6:	learn: 0.6420579	total: 356ms	remaining: 50.5s
7:	learn: 0.6367741	total: 405ms	remaining: 50.3s
8:	learn: 0.6312500	total: 458ms	remaining: 50.4s
9:	learn: 0.6263341	total: 509ms	remaining: 50.4s
10:	learn: 0.6217445	total: 555ms	remaining: 49.9s
11:	learn: 0.6174473	total: 603ms	remaining: 49.7s
12:	learn: 0.6133319	total: 655ms	remaining: 49.7s
13:	learn: 0.6095718	total: 706ms	remaining: 49.7s
14:	learn: 0.6060709	total: 759ms	remaining: 49.8s
15:	learn: 0.6028829	total: 839ms	remaining: 51.6s
16:	learn: 0.5998037	total: 918ms	remaining: 53.1s
17:	learn: 0.5967117	total: 992ms	remaining: 54.1s
18:	learn: 0.5940896	total: 1.06s	remaining: 54.8s
19:	learn: 

In [33]:
score = accuracy_score(y_test, y_pred)
score

0.7129880478087649